# Module 02: Modern SQL Mastery & Window Functions

Explore ranking functions, window frames, CTEs, and recursive hierarchy traversal in SQLite.


In [ ]:
import sqlite3
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
print('In-memory SQLite initialized.')


## 1. Schema Setup: Sales & Personnel


In [ ]:
cur.executescript('''
CREATE TABLE sales (
    id INTEGER PRIMARY KEY,
    seller TEXT,
    region TEXT,
    amount REAL,
    sale_date TEXT
);
INSERT INTO sales VALUES
    (1, 'Alice', 'North', 500, '2026-01-01'),
    (2, 'Bob', 'North', 300, '2026-01-02'),
    (3, 'Alice', 'North', 700, '2026-01-03'),
    (4, 'Charlie', 'South', 400, '2026-01-01'),
    (5, 'Diana', 'South', 600, '2026-01-02');
''')
conn.commit()


## 2. Ranking Functions: ROW_NUMBER(), RANK(), DENSE_RANK()


In [ ]:
cur.execute('''
SELECT seller, region, amount,
    ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC) as row_num,
    DENSE_RANK() OVER (PARTITION BY region ORDER BY amount DESC) as dense_rk
FROM sales
''')
for r in cur.fetchall():
    print(r)


## 3. Running Totals with Window Framing (ROWS vs RANGE)


In [ ]:
cur.execute('''
SELECT seller, sale_date, amount,
    SUM(amount) OVER (ORDER BY sale_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_sum
FROM sales
ORDER BY sale_date
''')
for r in cur.fetchall():
    print(r)


## 4. Lead and Lag: Temporal Delta Calculation


In [ ]:
cur.execute('''
SELECT seller, sale_date, amount,
    LAG(amount, 1) OVER (PARTITION BY seller ORDER BY sale_date) as prev_amount,
    amount - COALESCE(LAG(amount, 1) OVER (PARTITION BY seller ORDER BY sale_date), 0) as diff
FROM sales
''')
for r in cur.fetchall():
    print(r)


## 5. Recursive CTE: Hierarchical Organization Traversal


In [ ]:
cur.executescript('''
CREATE TABLE org (emp_id INT PRIMARY KEY, name TEXT, mgr_id INT);
INSERT INTO org VALUES (1, 'CEO', NULL), (2, 'VP Ops', 1), (3, 'Director', 2), (4, 'Lead', 3);
''')
cur.execute('''
WITH RECURSIVE Tree AS (
    SELECT emp_id, name, mgr_id, 0 as lvl, name as path FROM org WHERE mgr_id IS NULL
    UNION ALL
    SELECT o.emp_id, o.name, o.mgr_id, t.lvl + 1, t.path || ' -> ' || o.name
    FROM org o JOIN Tree t ON o.mgr_id = t.emp_id
)
SELECT lvl, name, path FROM Tree;
''')
for r in cur.fetchall():
    print(r)


## Summary

Window functions enable sophisticated analytical reporting without costly self-joins.
